In [ ]:
# ============================================================
# PHASE 4 GRU + RELU — MODIFIED STARTUP VERSION
# FIXES: Google Drive copy / transport endpoint error
# Uses Phase 3 directly from Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import gc
import json
import time
import pickle
import random
import warnings
import re
from glob import glob

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ============================================================
# CONFIG
# ============================================================

PHASE3_DRIVE_DIR = "/content/drive/MyDrive/Instacart/phase3_outputs_final"

# IMPORTANT: directly use Drive, no local copy
PHASE3_DIR = PHASE3_DRIVE_DIR

PHASE4_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable"
os.makedirs(PHASE4_DIR, exist_ok=True)

NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
VAL_SPLIT = 0.10
SEED = 42

INNER_BATCH_SIZE = 256
PRINT_EVERY_FILES = 25

EMBED_DIM_PRODUCT = 64
EMBED_DIM_AISLE = 8
EMBED_DIM_DEPT = 4
EMBED_DIM_DOW = 4
EMBED_DIM_HOUR = 4
TIME_FEATURE_DIM = 4
RNN_HIDDEN_DIM = 64
DENSE_DIM = 64
RECENCY_BETA_INIT = 0.10

MODEL_VARIANTS = [
    {
        "name": "Light_TimeAwareAttentionGRU_ReLU",
        "rnn_type": "GRU",
        "activation": "relu"
    }
]

FORCE_CONTINUE_TRAINING = True

# ============================================================
# HELPERS
# ============================================================

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def print_line():
    print("=" * 80)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print_line()
print("Using Phase 3 directly from Drive:")
print(PHASE3_DIR)

# ============================================================
# LOAD METADATA
# ============================================================

metadata_candidates = glob(os.path.join(PHASE3_DIR, "phase3_metadata*.pkl"))

if len(metadata_candidates) == 0:
    raise FileNotFoundError(f"No Phase 3 metadata found inside {PHASE3_DIR}")

stable_metadata = os.path.join(PHASE3_DIR, "phase3_metadata.pkl")

if os.path.exists(stable_metadata):
    metadata_path = stable_metadata
else:
    metadata_path = max(metadata_candidates, key=os.path.getmtime)

metadata = load_pickle(metadata_path)

print_line()
print("Loaded metadata from:", metadata_path)

for k, v in metadata.items():
    if isinstance(v, (int, float, str)):
        print(f"{k}: {v}")

RUN_ID = metadata["RUN_ID"]

print_line()
print("Metadata RUN_ID:", RUN_ID)

# ============================================================
# ROBUST REAL BATCH FILE DISCOVERY
# ============================================================

print_line()
print("Discovering real Phase 3 batch files...")

all_pkl_files = sorted(glob(os.path.join(PHASE3_DIR, "*.pkl")))

# Try metadata RUN_ID first
train_pattern_current = re.compile(rf"^train_batch_\d{{4}}_{RUN_ID}\.pkl$")
test_pattern_current  = re.compile(rf"^test_batch_\d{{4}}_{RUN_ID}\.pkl$")

raw_train_batch_files = [
    f for f in all_pkl_files
    if train_pattern_current.match(os.path.basename(f))
]

raw_test_batch_files = [
    f for f in all_pkl_files
    if test_pattern_current.match(os.path.basename(f))
]

print("Real train batches for metadata RUN_ID:", len(raw_train_batch_files))
print("Real test batches for metadata RUN_ID :", len(raw_test_batch_files))

# Fallback: find real batches from any RUN_ID
if len(raw_train_batch_files) == 0 or len(raw_test_batch_files) == 0:
    print("\nNo real batch files found for metadata RUN_ID.")
    print("Searching all real batch files from any RUN_ID...")

    train_pattern_any = re.compile(r"^train_batch_\d{4}_\d{8}_\d{6}\.pkl$")
    test_pattern_any  = re.compile(r"^test_batch_\d{4}_\d{8}_\d{6}\.pkl$")

    raw_train_batch_files = [
        f for f in all_pkl_files
        if train_pattern_any.match(os.path.basename(f))
    ]

    raw_test_batch_files = [
        f for f in all_pkl_files
        if test_pattern_any.match(os.path.basename(f))
    ]

if len(raw_train_batch_files) > 0:
    sample_name = os.path.basename(raw_train_batch_files[0])
    m = re.match(r"^train_batch_\d{4}_(\d{8}_\d{6})\.pkl$", sample_name)
    ACTUAL_BATCH_RUN_ID = m.group(1) if m else RUN_ID
else:
    ACTUAL_BATCH_RUN_ID = RUN_ID

print_line()
print("Metadata RUN_ID:", RUN_ID)
print("Actual batch RUN_ID used:", ACTUAL_BATCH_RUN_ID)
print("Final real train batch files:", len(raw_train_batch_files))
print("Final real test batch files :", len(raw_test_batch_files))
print("Expected train batches from metadata:", metadata.get("num_train_batches"))
print("Expected test batches from metadata :", metadata.get("num_test_batches"))

print("\nSample real train batch files:")
for f in raw_train_batch_files[:5]:
    print(os.path.basename(f))

print("\nSample real test batch files:")
for f in raw_test_batch_files[:5]:
    print(os.path.basename(f))

if len(raw_train_batch_files) == 0 or len(raw_test_batch_files) == 0:
    raise FileNotFoundError("No real train/test batch files found.")

# ============================================================
# VALIDATE REAL BATCH FILES
# ============================================================

REQUIRED_KEYS = {"Xp", "Xa", "Xd", "Xdow", "Xhr", "Xdays", "y"}

def validate_batch_files(file_list, label="train"):
    valid_files = []
    invalid_files = []

    for i, fpath in enumerate(file_list, start=1):
        try:
            obj = load_pickle(fpath)

            if isinstance(obj, dict) and REQUIRED_KEYS.issubset(set(obj.keys())):
                valid_files.append(fpath)
            else:
                invalid_files.append(
                    (
                        fpath,
                        list(obj.keys()) if isinstance(obj, dict) else str(type(obj))
                    )
                )

            del obj
            gc.collect()

        except Exception as e:
            invalid_files.append((fpath, str(e)))

        if i % 100 == 0 or i == len(file_list):
            print(f"{label}: checked {i}/{len(file_list)} files")

    return valid_files, invalid_files

train_batch_files, invalid_train_files = validate_batch_files(
    raw_train_batch_files,
    label="train"
)

test_batch_files, invalid_test_files = validate_batch_files(
    raw_test_batch_files,
    label="test"
)

print_line()
print("Valid train batch files:", len(train_batch_files))
print("Valid test batch files :", len(test_batch_files))
print("Invalid train files    :", len(invalid_train_files))
print("Invalid test files     :", len(invalid_test_files))

if len(train_batch_files) == 0 or len(test_batch_files) == 0:
    raise FileNotFoundError("No valid train/test batch files found after validation.")

# ============================================================
# TRAIN / VALIDATION SPLIT
# ============================================================

set_seed(SEED)

all_train_files = train_batch_files.copy()
random.shuffle(all_train_files)

val_count = max(1, int(len(all_train_files) * VAL_SPLIT))
val_files = all_train_files[:val_count]
train_files = all_train_files[val_count:]

print_line()
print("Train files:", len(train_files))
print("Val files  :", len(val_files))
print("Test files :", len(test_batch_files))

save_json(
    {
        "seed": SEED,
        "val_split": VAL_SPLIT,
        "num_train_files": len(train_files),
        "num_val_files": len(val_files),
        "num_test_files": len(test_batch_files),
        "metadata_run_id": RUN_ID,
        "actual_batch_run_id": ACTUAL_BATCH_RUN_ID
    },
    os.path.join(PHASE4_DIR, "split_info.json")
)

# ============================================================
# INSPECT ONE REAL BATCH
# ============================================================

sample_batch = load_pickle(train_files[0])

print_line()
print("Sample batch file:", os.path.basename(train_files[0]))
print("Sample batch keys:", sample_batch.keys())

for k, v in sample_batch.items():
    arr = np.array(v)
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")

print_line()
print("Xp   min/max:", np.min(sample_batch["Xp"]), np.max(sample_batch["Xp"]))
print("Xa   min/max:", np.min(sample_batch["Xa"]), np.max(sample_batch["Xa"]))
print("Xd   min/max:", np.min(sample_batch["Xd"]), np.max(sample_batch["Xd"]))
print("Xdow min/max:", np.min(sample_batch["Xdow"]), np.max(sample_batch["Xdow"]))
print("Xhr  min/max:", np.min(sample_batch["Xhr"]), np.max(sample_batch["Xhr"]))
print("y    min/max:", np.min(sample_batch["y"]), np.max(sample_batch["y"]))

print_line()
print("✅ Startup, batch discovery, validation, and split completed successfully.")

# ============================================================
# SAFE VOCAB SIZES
# Continue your existing GRU model code from here
# ============================================================

PRODUCT_VOCAB_SIZE = 25001
AISLE_VOCAB_SIZE   = 135
DEPT_VOCAB_SIZE    = 22
SAFE_MAX_DOW       = 7
SAFE_MAX_HOUR      = 24

print_line()
print("PRODUCT_VOCAB_SIZE:", PRODUCT_VOCAB_SIZE)
print("AISLE_VOCAB_SIZE  :", AISLE_VOCAB_SIZE)
print("DEPT_VOCAB_SIZE   :", DEPT_VOCAB_SIZE)
print("SAFE_MAX_DOW      :", SAFE_MAX_DOW)
print("SAFE_MAX_HOUR     :", SAFE_MAX_HOUR)

Mounted at /content/drive
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Using device: cuda
Using Phase 3 directly from Drive:
/content/drive/MyDrive/Instacart/phase3_outputs_final
Loaded metadata from: /content/drive/MyDrive/Instacart/phase3_outputs_final/phase3_metadata.pkl
RUN_ID: 20260429_204444
MAX_LEN: 50
CHUNK_SIZE: 100
num_train_users: 164331
num_test_users: 41083
num_train_batches: 1644
num_test_batches: 411
product_vocab_size: 25001
aisle_vocab_size: 134
dept_vocab_size: 21
sample_train_batch_file: train_batch_0001_20260422_185902.pkl
Metadata RUN_ID: 20260429_204444
Discovering real Phase 3 batch files...
Real train batches for metadata RUN_ID: 0
Real test batches for metadata RUN_ID : 0

No real batch files found for metadata RUN_ID.
Searching all real batch files from any RUN_ID...
Metadata RUN_ID: 20260429_204444
Actual batch RUN_ID used: 20260422_185902
Final real train batch files: 1644
Final real test batch files : 411
Expected train batches from meta

In [ ]:
import os
import torch
import pandas as pd

MODEL_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable/Light_TimeAwareAttentionGRU_ReLU"

latest_ckpt = os.path.join(MODEL_DIR, "checkpoint_latest.pt")
best_ckpt   = os.path.join(MODEL_DIR, "checkpoint_best.pt")
history_csv = os.path.join(MODEL_DIR, "training_history.csv")
done_flag   = os.path.join(MODEL_DIR, "TRAINING_DONE.flag")

print("MODEL DIR:", MODEL_DIR)
print("\n--- FILE CHECK ---")
print("Latest checkpoint :", os.path.exists(latest_ckpt))
print("Best checkpoint   :", os.path.exists(best_ckpt))
print("History CSV       :", os.path.exists(history_csv))
print("Done flag         :", os.path.exists(done_flag))


# ------------------------------------------------------------
# IF LATEST CHECKPOINT EXISTS → SHOW PROGRESS
# ------------------------------------------------------------
if os.path.exists(latest_ckpt):
    ckpt = torch.load(latest_ckpt, map_location="cpu")

    print("\n===== LATEST CHECKPOINT =====")
    print("Last trained epoch:", ckpt.get("epoch"))
    print("Best val F1 so far:", ckpt.get("best_val_f1"))


# ------------------------------------------------------------
# IF BEST CHECKPOINT EXISTS → SHOW BEST MODEL
# ------------------------------------------------------------
if os.path.exists(best_ckpt):
    ckpt = torch.load(best_ckpt, map_location="cpu")

    print("\n===== BEST CHECKPOINT =====")
    print("Best epoch:", ckpt.get("epoch"))
    print("Best val F1:", ckpt.get("best_val_f1"))


# ------------------------------------------------------------
# IF HISTORY EXISTS → SHOW LAST ROW
# ------------------------------------------------------------
if os.path.exists(history_csv):
    df = pd.read_csv(history_csv)

    print("\n===== LAST ROW (FINAL EPOCH) =====")
    display(df.tail(1))

    print("\n===== BEST ROW (MAX VAL F1) =====")
    display(df.loc[df["val_f1_weighted"].idxmax()])

MODEL DIR: /content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable/Light_TimeAwareAttentionGRU_ReLU

--- FILE CHECK ---
Latest checkpoint : True
Best checkpoint   : True
History CSV       : True
Done flag         : True

===== LATEST CHECKPOINT =====
Last trained epoch: 20
Best val F1 so far: 0.013280621842388365

===== BEST CHECKPOINT =====
Best epoch: 16
Best val F1: 0.013280621842388365

===== LAST ROW (FINAL EPOCH) =====


,epoch,train_loss,val_loss,train_accuracy,train_precision_weighted,train_recall_weighted,train_f1_weighted,val_accuracy,val_precision_weighted,val_recall_weighted,val_f1_weighted,epoch_time_sec
19,20,7.486216,7.323169,0.032036,0.01849,0.032036,0.013357,0.036352,0.020582,0.036352,0.013239,1620.361952



===== BEST ROW (MAX VAL F1) =====


,15
epoch,16.000000
train_loss,7.491733
val_loss,7.330181
train_accuracy,0.032677
train_precision_weighted,0.018262
train_recall_weighted,0.032677
train_f1_weighted,0.013365
val_accuracy,0.036740
val_precision_weighted,0.020476
val_recall_weighted,0.036740


In [ ]:
# ============================================================
# STANDALONE TOP-K EVALUATION
# MODEL: Light_TimeAwareAttentionGRU_ReLU
# ============================================================

import os
import gc
import json
import pickle
import re
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

PHASE3_DIR = "/content/drive/MyDrive/Instacart/phase3_outputs_final"

MODEL_NAME = "Light_TimeAwareAttentionGRU_ReLU"
MODEL_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable/Light_TimeAwareAttentionGRU_ReLU"
BEST_CKPT = os.path.join(MODEL_DIR, "checkpoint_best.pt")

OUTPUT_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable"

PRODUCT_VOCAB_SIZE = 25001
AISLE_VOCAB_SIZE = 135
DEPT_VOCAB_SIZE = 22
SAFE_MAX_DOW = 7
SAFE_MAX_HOUR = 24

TOPK_VALUES = [5, 10]
INNER_BATCH_SIZE = 64

print("Best checkpoint exists:", os.path.exists(BEST_CKPT))

if not os.path.exists(BEST_CKPT):
    raise FileNotFoundError(f"Checkpoint not found: {BEST_CKPT}")


# ============================================================
# HELPERS
# ============================================================

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def load_batch_tensors(batch_path, device=DEVICE):
    batch = load_pickle(batch_path)

    return {
        "Xp": torch.tensor(batch["Xp"], dtype=torch.long, device=device),
        "Xa": torch.tensor(batch["Xa"], dtype=torch.long, device=device),
        "Xd": torch.tensor(batch["Xd"], dtype=torch.long, device=device),
        "Xdow": torch.tensor(batch["Xdow"], dtype=torch.long, device=device),
        "Xhr": torch.tensor(batch["Xhr"], dtype=torch.long, device=device),
        "Xdays": torch.tensor(batch["Xdays"], dtype=torch.float32, device=device),
        "y": torch.tensor(batch["y"], dtype=torch.long, device=device)
    }

def iterate_inner_batches(batch_tensors, inner_batch_size=64):
    n = batch_tensors["y"].shape[0]

    for start in range(0, n, inner_batch_size):
        end = min(start + inner_batch_size, n)

        yield {
            "Xp": batch_tensors["Xp"][start:end],
            "Xa": batch_tensors["Xa"][start:end],
            "Xd": batch_tensors["Xd"][start:end],
            "Xdow": batch_tensors["Xdow"][start:end],
            "Xhr": batch_tensors["Xhr"][start:end],
            "Xdays": batch_tensors["Xdays"][start:end],
            "y": batch_tensors["y"][start:end]
        }


# ============================================================
# MODEL ARCHITECTURE — MATCHES TRAINED GRU + RELU MODEL
# ============================================================

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_outputs, mask=None):
        e = torch.tanh(self.attn(rnn_outputs))
        scores = self.score(e).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), rnn_outputs).squeeze(1)

        return context, weights


class TimeAwareAttentionGRUReLU(nn.Module):
    def __init__(self):
        super().__init__()

        self.product_emb = nn.Embedding(PRODUCT_VOCAB_SIZE, 64, padding_idx=0)
        self.aisle_emb = nn.Embedding(AISLE_VOCAB_SIZE, 8, padding_idx=0)
        self.dept_emb = nn.Embedding(DEPT_VOCAB_SIZE, 4, padding_idx=0)
        self.dow_emb = nn.Embedding(SAFE_MAX_DOW, 4, padding_idx=0)
        self.hour_emb = nn.Embedding(SAFE_MAX_HOUR, 4, padding_idx=0)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, 4),
            nn.ReLU(),
            nn.Linear(4, 4)
        )

        self.recency_beta = nn.Parameter(torch.tensor(0.10, dtype=torch.float32))

        input_dim = 64 + 8 + 4 + 4 + 4 + 4 + 1

        self.rnn = nn.GRU(
            input_size=input_dim,
            hidden_size=64,
            batch_first=True
        )

        self.attention = AttentionLayer(64)

        self.fc1 = nn.Linear(64, 64)
        self.dropout = nn.Dropout(0.2)
        self.activation = nn.ReLU()

        self.fc_out = nn.Linear(64, PRODUCT_VOCAB_SIZE)

    def forward(self, Xp, Xa, Xd, Xdow, Xhr, Xdays):
        p_emb = self.product_emb(Xp)
        a_emb = self.aisle_emb(Xa)
        d_emb = self.dept_emb(Xd)
        dow_emb = self.dow_emb(Xdow)
        hr_emb = self.hour_emb(Xhr)

        xdays_log = torch.log1p(Xdays)
        xdays_norm = xdays_log / (xdays_log.max().detach() + 1e-8)

        gap_feature = xdays_norm.unsqueeze(-1)
        time_encoded = self.time_mlp(gap_feature)

        x = torch.cat(
            [p_emb, a_emb, d_emb, dow_emb, hr_emb, time_encoded, gap_feature],
            dim=-1
        )

        beta = torch.clamp(self.recency_beta, min=0.0)
        recency_weight = torch.exp(-beta * xdays_norm).unsqueeze(-1)
        x = x * recency_weight

        mask = (Xp != 0).long()

        rnn_out, _ = self.rnn(x)
        context, attn_weights = self.attention(rnn_out, mask=mask)

        h = self.fc1(context)
        h = self.activation(h)
        h = self.dropout(h)

        logits = self.fc_out(h)

        return logits, attn_weights


# ============================================================
# LOAD BEST CHECKPOINT
# ============================================================

model = TimeAwareAttentionGRUReLU().to(DEVICE)

checkpoint = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

best_epoch = checkpoint.get("epoch")
best_val_f1 = checkpoint.get("best_val_f1")

print("Loaded best checkpoint.")
print("Best epoch:", best_epoch)
print("Best val F1:", best_val_f1)


# ============================================================
# FIND REAL TEST BATCH FILES
# ============================================================

all_pkl_files = sorted(glob(os.path.join(PHASE3_DIR, "*.pkl")))

test_batch_files = [
    f for f in all_pkl_files
    if re.match(r".*test_batch_\d{4}_\d{8}_\d{6}\.pkl$", f)
]

print("Test batch files found:", len(test_batch_files))

if len(test_batch_files) == 0:
    raise FileNotFoundError("No real test batch files found.")


# ============================================================
# TOP-K EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_topk(model, batch_files, k_values=[5, 10], inner_batch_size=64):
    model.eval()

    total = 0
    hits = {k: 0 for k in k_values}

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size):
            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            y_true = mini_batch["y"].view(-1, 1)

            for k in k_values:
                topk_preds = torch.topk(logits, k=k, dim=1).indices
                hits[k] += (topk_preds == y_true).any(dim=1).sum().item()

            total += y_true.size(0)

            del mini_batch, logits, y_true
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % 25 == 0 or (file_idx + 1) == len(batch_files):
            print(f"Top-K progress: {file_idx + 1}/{len(batch_files)} test files processed")

    results = {"total_test_samples": total}

    for k in k_values:
        hit_rate = hits[k] / total
        results[f"hit_rate@{k}"] = hit_rate
        results[f"recall@{k}"] = hit_rate
        results[f"precision@{k}"] = hit_rate / k

    return results


# ============================================================
# RUN TOP-K
# ============================================================

topk_metrics = evaluate_topk(
    model=model,
    batch_files=test_batch_files,
    k_values=TOPK_VALUES,
    inner_batch_size=INNER_BATCH_SIZE
)

topk_result = {
    "model_name": MODEL_NAME,
    "best_epoch": best_epoch,
    "best_val_f1": best_val_f1,
    **topk_metrics
}

topk_df = pd.DataFrame([topk_result])
display(topk_df)


# ============================================================
# SAVE RESULTS
# ============================================================

topk_csv = os.path.join(OUTPUT_DIR, "phase4_gru_relu_topk_results.csv")
topk_json = os.path.join(OUTPUT_DIR, "phase4_gru_relu_topk_results.json")

topk_df.to_csv(topk_csv, index=False)
save_json([topk_result], topk_json)

print("\nSaved Top-K results to:")
print(topk_csv)
print(topk_json)

Device: cuda
Best checkpoint exists: True
Loaded best checkpoint.
Best epoch: 16
Best val F1: 0.013280621842388365
Test batch files found: 411
Top-K progress: 25/411 test files processed
Top-K progress: 50/411 test files processed
Top-K progress: 75/411 test files processed
Top-K progress: 100/411 test files processed
Top-K progress: 125/411 test files processed
Top-K progress: 150/411 test files processed
Top-K progress: 175/411 test files processed
Top-K progress: 200/411 test files processed
Top-K progress: 225/411 test files processed
Top-K progress: 250/411 test files processed
Top-K progress: 275/411 test files processed
Top-K progress: 300/411 test files processed
Top-K progress: 325/411 test files processed
Top-K progress: 350/411 test files processed
Top-K progress: 375/411 test files processed
Top-K progress: 400/411 test files processed
Top-K progress: 411/411 test files processed


,model_name,best_epoch,best_val_f1,total_test_samples,hit_rate@5,recall@5,precision@5,hit_rate@10,recall@10,precision@10
0,Light_TimeAwareAttentionGRU_ReLU,16,0.013281,6348182,0.100307,0.100307,0.020061,0.145853,0.145853,0.014585



Saved Top-K results to:
/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable/phase4_gru_relu_topk_results.csv
/content/drive/MyDrive/Instacart/phase4_outputs_gru_relu_stable/phase4_gru_relu_topk_results.json
